# Price Data Imputation

1. MAR diagnostic via logistic regression of missingness indicator on observed features
2. Predictive imputation of `price_per_square_meter` using Random Forest on complete cases
3. Delta-adjustment sensitivity table for MNAR robustness


In [22]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from scipy import stats


In [23]:
df = pd.read_excel('../data/properties_modified.xlsx')

df_work = df.copy()
original_dtypes = df.dtypes

# Encode categoricals (NaN → 'Unknown' so encoder has no NaN)
le_dict = {}
for col in df_work.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_work[col] = le.fit_transform(df_work[col].fillna('Unknown'))
    le_dict[col] = le

available_vars = [c for c in df.columns if c != 'price_per_square_meter']
X = df_work[available_vars].copy()
y = df_work['price_per_square_meter'].copy()

missing_mask = y.isna()
observed_prices = y.dropna()

print(f"Rows: {len(df):,}  |  Missing prices: {missing_mask.sum():,} ({missing_mask.mean():.1%})")


Rows: 8,089  |  Missing prices: 1,140 (14.1%)


In [24]:
# Median-impute features for model input only (original X is unchanged)
X_filled = pd.DataFrame(
    SimpleImputer(strategy='median').fit_transform(X),
    columns=X.columns, index=X.index
)

complete_mask = y.notna()
lower_bound = observed_prices.quantile(0.05)
upper_bound = observed_prices.quantile(0.95)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_filled[complete_mask], y[complete_mask])

cv_r2 = cross_val_score(rf, X_filled[complete_mask], y[complete_mask], cv=5, scoring='r2', n_jobs=-1).mean()

y_imputed = y.copy()
y_imputed[missing_mask] = rf.predict(X_filled[missing_mask]).clip(lower_bound, upper_bound)

print(f"Imputed {missing_mask.sum():,} prices  |  CV R²: {cv_r2:.4f}")
print(f"Mean: {y_imputed[missing_mask].mean():,.0f}  |  Range: [{y_imputed[missing_mask].min():,.0f}, {y_imputed[missing_mask].max():,.0f}]")


Imputed 1,140 prices  |  CV R²: 0.6876
Mean: 18,539  |  Range: [12,108, 27,321]


In [25]:
def mar_logistic_test(X, missing_mask, max_features=20):
    """Logistic regression of missingness indicator on observed features.
    Returns omnibus LR p-value and McFadden pseudo-R²."""
    X_filled = pd.DataFrame(
        SimpleImputer(strategy='median').fit_transform(X),
        columns=X.columns, index=X.index
    ).iloc[:, :max_features]

    is_missing = missing_mask.astype(int)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_filled, is_missing)

    p_hat  = lr.predict_proba(X_filled)[:, 1]
    p_null = is_missing.mean()
    ll_model = np.sum(is_missing * np.log(p_hat + 1e-15) + (1 - is_missing) * np.log(1 - p_hat + 1e-15))
    ll_null  = np.sum(is_missing * np.log(p_null)        + (1 - is_missing) * np.log(1 - p_null))

    pseudo_r2 = 1 - ll_model / ll_null
    p_value   = stats.chi2.sf(-2 * (ll_null - ll_model), df=X_filled.shape[1])
    coefs     = dict(zip(X_filled.columns, lr.coef_[0]))
    return {'pseudo_r2': pseudo_r2, 'p_value': p_value, 'coef': coefs}


mar_result = mar_logistic_test(X, missing_mask)

print(f"MAR Diagnostic  |  pseudo-R²: {mar_result['pseudo_r2']:.4f}  |  p-value: {mar_result['p_value']:.4e}")
print("\nTop predictors of missingness:")
for feat, coef in sorted(mar_result['coef'].items(), key=lambda x: abs(x[1]), reverse=True)[:5]:
    print(f"  {feat:<35s}  {coef:+.4f}")


MAR Diagnostic  |  pseudo-R²: 0.5357  |  p-value: 0.0000e+00

Top predictors of missingness:
  market                               -3.8269
  dist_to_metro                        +2.9258
  dist_to_center                       +2.9216
  district_group_Włochy_Praga Południe  -2.5385
  district_group_Ursynów_Praga Północ  +1.9377


In [26]:
p_value, pseudo_r2 = mar_result['p_value'], mar_result['pseudo_r2']

if p_value < 0.05 and pseudo_r2 > 0.02:
    assumption, confidence = "MAR",  "High"
    reason = f"Features significantly predict missingness (p={p_value:.2e}, pseudo-R²={pseudo_r2:.3f})"
elif p_value < 0.05:
    assumption, confidence = "MAR",  "Moderate"
    reason = f"Weak prediction of missingness (p={p_value:.2e}, pseudo-R²={pseudo_r2:.3f})"
else:
    assumption, confidence = "MCAR", "High"
    reason = f"Features do not predict missingness (p={p_value:.2e})"

# Note: MNAR cannot be confirmed statistically — requires domain knowledge.
print(f"Assumption: {assumption}  |  Confidence: {confidence}")
print(f"Reason    : {reason}")


Assumption: MAR  |  Confidence: High
Reason    : Features significantly predict missingness (p=0.00e+00, pseudo-R²=0.536)


In [27]:
def restore_dtypes(df_in, original_dtypes, le_dict):
    """Decode label-encoded columns and restore original dtypes."""
    out = df_in.copy()
    for col in out.columns:
        if col in le_dict:
            out[col] = le_dict[col].inverse_transform(
                out[col].round().astype(int).clip(0, len(le_dict[col].classes_) - 1)
            )
        elif col in original_dtypes.index and pd.api.types.is_integer_dtype(original_dtypes[col]):
            out[col] = out[col].round().astype(original_dtypes[col])
    return out


final_df_work = df_work.copy()
final_df_work['price_per_square_meter'] = y_imputed
final_dataset = restore_dtypes(final_df_work, original_dtypes, le_dict)

print(f"Shape: {final_dataset.shape}  |  Missing: {final_dataset.isna().sum().sum()}")
print(f"Imputed prices — Mean: {y_imputed[missing_mask].mean():,.0f}  Std: {y_imputed[missing_mask].std():,.0f}")

# Delta-adjustment sensitivity (MNAR robustness)
print("\n  δ%   |    Mean    |    Std     |    Min     |    Max")
print("  " + "-" * 55)
for delta in [-20, -10, -5, 0, 5, 10, 20]:
    s = y_imputed[missing_mask] * (1 + delta / 100)
    mark = " ◄" if delta == 0 else ""
    print(f"  {delta:+4d}% | {s.mean():>10,.0f} | {s.std():>10,.0f} | {s.min():>10,.0f} | {s.max():>10,.0f}{mark}")


Shape: (8089, 21)  |  Missing: 0
Imputed prices — Mean: 18,539  Std: 2,884

  δ%   |    Mean    |    Std     |    Min     |    Max
  -------------------------------------------------------
   -20% |     14,831 |      2,307 |      9,686 |     21,857
   -10% |     16,685 |      2,596 |     10,897 |     24,589
    -5% |     17,612 |      2,740 |     11,503 |     25,955
    +0% |     18,539 |      2,884 |     12,108 |     27,321 ◄
    +5% |     19,466 |      3,028 |     12,714 |     28,687
   +10% |     20,393 |      3,172 |     13,319 |     30,053
   +20% |     22,247 |      3,461 |     14,530 |     32,786


In [28]:
final_dataset.to_excel('../data/abt.xlsx', index=False)